In [57]:
import pandas as pd


In [58]:
mtr_lines_fares=pd.read_csv("../Data/MTR/mtr_lines_fares.csv")
mtr_lines_fares.head()

,id,SRC_STATION_NAME,SRC_STATION_ID,DEST_STATION_NAME,DEST_STATION_ID,OCT_ADT_FARE,OCT_STD_FARE,OCT_JOYYOU_SIXTY_FARE,SINGLE_ADT_FARE,OCT_CON_CHILD_FARE,OCT_CON_ELDERLY_FARE,OCT_CON_PWD_FARE,SINGLE_CON_CHILD_FARE,SINGLE_CON_ELDERLY_FARE
0,1,Central,1,Cent ·ral,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,Central,1,Admiralty,2,4.9,3.2,2.0,5.0,3.2,2.0,2.0,3.5,3.5
2,3,Central,1,Tsim Sha Tsui,3,10.6,5.4,2.0,12.5,5.4,2.0,2.0,5.5,5.5
3,4,Central,1,Jordan,4,10.6,5.4,2.0,12.5,5.4,2.0,2.0,5.5,5.5
4,5,Central,1,Yau Ma Tei,5,13.2,6.5,2.0,15.5,6.5,2.0,2.0,6.5,6.5


In [59]:
MTR_DISTRICT=pd.read_excel("../Data/MTR/MTR_DISTRICT.xlsx")
MTR_DISTRICT.head()

,Line_Code,Station_ID,Chinese_Name,Sequence,DISTRICT,Direction
0,ISL,1,中環,13,中西區,DT
1,TWL,1,中環,16,中西區,DT
2,ISL,2,金鐘,12,中西區,DT
3,EAL,2,金鐘,14,中西區,DT
4,SIL,2,金鐘,5,中西區,DT


In [60]:
MTR_COORDINATES=pd.read_excel("../Data/MTR/MTR_STATIONS_COORDINATES.xlsx",engine="openpyxl")
MTR_COORDINATES.head()

,Line Code,Direction,Station Code,Station ID,Chinese Name,English Name,Sequence,Lat,Long
0,ISL,DT,CEN,1,中環,Central,13,22.2820,114.1576
1,TWL,DT,CEN,1,中環,Central,16,22.2820,114.1576
2,WLK,--,CEN,1,中環,Central,1,22.2820,114.1576
3,ISL,DT,ADM,2,金鐘,Admiralty,12,22.2788,114.1646
4,TWL,DT,ADM,2,金鐘,Admiralty,15,22.2788,114.1646


In [61]:
mtr_src_fares=mtr_lines_fares.merge(
    MTR_DISTRICT[['Station_ID','DISTRICT','Chinese_Name','Sequence','Line_Code']],
    left_on=['SRC_STATION_ID'],
    right_on=['Station_ID']
    ,how='left').rename(columns={'DISTRICT':'ON_DISTRICT','Chinese_Name':'ON_NAME','Sequence':'ON_SEQUENCE','Line_Code':'ON_LINE'})
mtr_dest_fares=mtr_src_fares.merge(
    MTR_DISTRICT[['Station_ID','DISTRICT','Chinese_Name','Sequence','Line_Code']],
    left_on=['DEST_STATION_ID'],
    right_on=['Station_ID']
    ,how='left'
).rename(columns={'DISTRICT':'OFF_DISTRICT','Chinese_Name':'OFF_NAME','Sequence':'OFF_SEQUENCE','Line_Code':'OFF_LINE'})
mtr_dest_fares=mtr_dest_fares.drop(columns=['Station_ID_x','Station_ID_y'])

In [62]:
mtr_dest_fares=mtr_dest_fares.drop(columns=['OCT_STD_FARE','OCT_STD_FARE','OCT_JOYYOU_SIXTY_FARE','SINGLE_ADT_FARE','OCT_CON_CHILD_FARE','OCT_CON_ELDERLY_FARE','OCT_CON_PWD_FARE','SINGLE_CON_CHILD_FARE','SINGLE_CON_ELDERLY_FARE'],errors='ignore')
mtr_dest_fares=mtr_dest_fares.rename(columns={'OCT_ADT_FARE':'Price'})
mtr_dest_fares=mtr_dest_fares[mtr_dest_fares['Price']>0]
mtr_dest_fares=mtr_dest_fares.groupby(['SRC_STATION_ID', 'DEST_STATION_ID'], as_index=False).first()
mtr_dest_fares.dropna(subset=['ON_DISTRICT'], inplace=True)

In [63]:
mtr_dest_fares.head()

,SRC_STATION_ID,DEST_STATION_ID,id,SRC_STATION_NAME,DEST_STATION_NAME,Price,ON_DISTRICT,ON_NAME,ON_SEQUENCE,ON_LINE,OFF_DISTRICT,OFF_NAME,OFF_SEQUENCE,OFF_LINE
0,1,2,2,Central,Admiralty,4.9,中西區,中環,13.0,ISL,中西區,金鐘,12.0,ISL
1,1,3,3,Central,Tsim Sha Tsui,10.6,中西區,中環,13.0,ISL,油尖旺,尖沙咀,14.0,TWL
2,1,4,4,Central,Jordan,10.6,中西區,中環,13.0,ISL,油尖旺,佐敦,13.0,TWL
3,1,5,5,Central,Yau Ma Tei,13.2,中西區,中環,13.0,ISL,油尖旺,油麻地,15.0,KTL
4,1,6,6,Central,Mong Kok,13.2,中西區,中環,13.0,ISL,油尖旺,旺角,14.0,KTL


### Add coordinates

In [64]:
mtr_src_coord=mtr_dest_fares.merge(
    MTR_COORDINATES[['Station ID','Lat','Long']],
    left_on=['SRC_STATION_ID'],
    right_on=['Station ID'],
    how='left'
).rename(columns={'Lat':'SRC_LAT','Long':'SRC_LONG'})

mtr_dest_coord=mtr_src_coord.merge(
    MTR_COORDINATES[['Station ID','Lat','Long']],
    left_on=['DEST_STATION_ID'],
    right_on=['Station ID'],
    how='left'
).rename(columns={'Lat':'DEST_LAT','Long':'DEST_LONG'})
mtr_dest_coord=mtr_dest_coord.drop(columns=['Station ID_x','Station ID_y'])
mtr_dest_coord=mtr_dest_coord.drop_duplicates()
mtr_dest_coord.head()

,SRC_STATION_ID,DEST_STATION_ID,id,SRC_STATION_NAME,DEST_STATION_NAME,Price,ON_DISTRICT,ON_NAME,ON_SEQUENCE,ON_LINE,OFF_DISTRICT,OFF_NAME,OFF_SEQUENCE,OFF_LINE,SRC_LAT,SRC_LONG,DEST_LAT,DEST_LONG
0,1,2,2,Central,Admiralty,4.9,中西區,中環,13.0,ISL,中西區,金鐘,12.0,ISL,22.282,114.1576,22.2788,114.1646
9,1,3,3,Central,Tsim Sha Tsui,10.6,中西區,中環,13.0,ISL,油尖旺,尖沙咀,14.0,TWL,22.282,114.1576,22.2973,114.1722
12,1,4,4,Central,Jordan,10.6,中西區,中環,13.0,ISL,油尖旺,佐敦,13.0,TWL,22.282,114.1576,22.3049,114.1718
15,1,5,5,Central,Yau Ma Tei,13.2,中西區,中環,13.0,ISL,油尖旺,油麻地,15.0,KTL,22.282,114.1576,22.3129,114.1707
21,1,6,6,Central,Mong Kok,13.2,中西區,中環,13.0,ISL,油尖旺,旺角,14.0,KTL,22.282,114.1576,22.3191,114.1694


In [65]:
mtr_dest_coord.to_excel("../Data/MTR/MTR_Conversion.xlsx", engine='openpyxl')

### Select route within districts

In [66]:
Routes_within_district=mtr_dest_coord[mtr_dest_coord['ON_DISTRICT']==mtr_dest_coord['OFF_DISTRICT']]

Routes_within_district.head()

,SRC_STATION_ID,DEST_STATION_ID,id,SRC_STATION_NAME,DEST_STATION_NAME,Price,ON_DISTRICT,ON_NAME,ON_SEQUENCE,ON_LINE,OFF_DISTRICT,OFF_NAME,OFF_SEQUENCE,OFF_LINE,SRC_LAT,SRC_LONG,DEST_LAT,DEST_LONG
0,1,2,2,Central,Admiralty,4.9,中西區,中環,13.0,ISL,中西區,金鐘,12.0,ISL,22.282,114.1576,22.2788,114.1646
99,1,26,26,Central,Sheung Wan,4.9,中西區,中環,13.0,ISL,中西區,上環,14.0,ISL,22.282,114.1576,22.2862,114.1518
243,1,81,83,Central,Sai Ying Pun,4.9,中西區,中環,13.0,ISL,中西區,西營盤,15.0,ISL,22.282,114.1576,22.2856,114.1427
246,1,82,84,Central,HKU,5.9,中西區,中環,13.0,ISL,中西區,香港大學,16.0,ISL,22.282,114.1576,22.2841,114.1356
249,1,83,85,Central,Kennedy Town,5.9,中西區,中環,13.0,ISL,中西區,堅尼地城,17.0,ISL,22.282,114.1576,22.2812,114.1285


In [67]:
from math import radians, sin, cos, sqrt, atan2

# Haversine 公式计算直线距离
def haversine(lat1, lon1, lat2, lon2):
    # 将经纬度转换为弧度
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    
    # 计算差值
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    # Haversine 公式
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    
    # 地球半径（公里）
    R = 6371.0
    distance = R * c
    return distance*1000


# 计算距离并存储到新列
Routes_within_district['DISTANCE'] = Routes_within_district.apply(
    lambda row: haversine(row['SRC_LAT'], row['SRC_LONG'], row['DEST_LAT'], row['DEST_LONG']),
    axis=1
)

# 打印结果
Routes_within_district=Routes_within_district.dropna(subset=['DISTANCE'], how='all')
Routes_within_district.head()

C:\Users\非衣\AppData\Local\Temp\ipykernel_33876\545833676.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Routes_within_district['DISTANCE'] = Routes_within_district.apply(


,SRC_STATION_ID,DEST_STATION_ID,id,SRC_STATION_NAME,DEST_STATION_NAME,Price,ON_DISTRICT,ON_NAME,ON_SEQUENCE,ON_LINE,OFF_DISTRICT,OFF_NAME,OFF_SEQUENCE,OFF_LINE,SRC_LAT,SRC_LONG,DEST_LAT,DEST_LONG,DISTANCE
0,1,2,2,Central,Admiralty,4.9,中西區,中環,13.0,ISL,中西區,金鐘,12.0,ISL,22.282,114.1576,22.2788,114.1646,803.350860
99,1,26,26,Central,Sheung Wan,4.9,中西區,中環,13.0,ISL,中西區,上環,14.0,ISL,22.282,114.1576,22.2862,114.1518,757.782053
243,1,81,83,Central,Sai Ying Pun,4.9,中西區,中環,13.0,ISL,中西區,西營盤,15.0,ISL,22.282,114.1576,22.2856,114.1427,1584.469234
246,1,82,84,Central,HKU,5.9,中西區,中環,13.0,ISL,中西區,香港大學,16.0,ISL,22.282,114.1576,22.2841,114.1356,2275.616627
249,1,83,85,Central,Kennedy Town,5.9,中西區,中環,13.0,ISL,中西區,堅尼地城,17.0,ISL,22.282,114.1576,22.2812,114.1285,2995.483327


In [68]:
district_total_distance = Routes_within_district.groupby("ON_DISTRICT")["DISTANCE"].transform("sum")

# 计算成本指数
Routes_within_district["COST_INDEX"] = (Routes_within_district["Price"] * Routes_within_district["DISTANCE"]) / district_total_distance

In [69]:
Selected_Routes=Routes_within_district.copy()
Selected_Routes.to_excel("../Data/MTR/Selected_Routes.xlsx", engine="openpyxl")


In [70]:
district_mean_cost = Selected_Routes.groupby('ON_DISTRICT')['COST_INDEX'].mean().reset_index()
district_mean_cost['COST_INDEX']=district_mean_cost['COST_INDEX']*100

district_mean_cost.rename(columns={'COST_INDEX': 'MEAN_COST_INDEX(%)'}, inplace=True)
district_counts = Selected_Routes.groupby('ON_DISTRICT').size().reset_index(name='NUMBER_ROUTES')

# Merge the counts into the district_mean_cost DataFrame
district_mean_cost = district_mean_cost.merge(district_counts, on='ON_DISTRICT')

print(district_mean_cost)

   ON_DISTRICT  MEAN_COST_INDEX(%)  NUMBER_ROUTES
0          中西區           13.801324             40
1          九龍城           21.584822             30
2           元朗           49.355415             12
3           北區          149.806994             12
4           南區           43.187172             12
5           屯門          265.000000              2
6           東區           11.090404             56
7           沙田            4.075188            156
8          油尖旺            6.326943             88
9          深水埗           13.956012             42
10          灣仔           81.666667              6
11          荃灣           81.666667              6
12          葵青           40.833333             12
13          西貢           42.912451             12
14          觀塘           26.239159             20
15          離島          215.192712              6
16         黃大仙          245.000000              2


In [55]:
district_mean_cost.to_excel("../Data/MTR/mtr_18_index.xlsx", engine="openpyxl")